# Demo 8 — Un notch e un passa-basso sullo stesso disturbo

Al primo avvio usare **Run → Run All Cells** (▶▶): la preparazione carica `audiolab`. Poi modificare la cella della demo e rieseguirla con **Shift + Invio**. Ogni riquadro è indipendente.

Le funzioni `al.…` sono quelle della libreria del corso. La prima esecuzione richiede il download del runtime e delle dipendenze.

In [ ]:
# Preparazione automatica: la stessa libreria audiolab dei laboratori.
import sys
from pathlib import Path
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["numpy", "scipy", "soundfile", "matplotlib", "plotly==6.3.0", "nbformat>=5.10"])
    from pyodide.http import pyfetch
    from js import location
    import io, zipfile
    ROOT = Path("/home/pyodide/infomm-3485c94980f2f6df")
    if not (ROOT / "lab-2026/audiolab/__init__.py").exists():
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        url = str(location.origin) + prefisso + "/_jupyterlite/files/audiolab-course.zip?v=3485c94980f2f6df"
        risposta = await pyfetch(url)
        if not risposta.ok:
            raise RuntimeError("Impossibile caricare audiolab: " + str(risposta.status))
        with zipfile.ZipFile(io.BytesIO(await risposta.bytes())) as archivio:
            archivio.extractall(ROOT)
else:
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "lab-2026/audiolab").is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError("Avviare Jupyter dalla cartella InfoMM-local del corso.")
sys.path.insert(0, str(ROOT / "lab-2026"))
import audiolab as al
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import signal
from IPython.display import Audio, Markdown, display
import time
pio.renderers.default = "plotly_mimetype"
AUDIO = al.AUDIO_DIR
DEMO_ASSETS = ROOT / "lab-2026/interattivo/assets"
SR = FS = 44100
N_FFT, HOP = 2048, 512
BLU, ARANCIO, VERDE = "#1766A5", "#D86822", "#277668"
print("audiolab", al.__version__, "— ambiente pronto")


In [ ]:
voce, sr = al.load_audio(AUDIO / "speech1.wav", sr=SR, seconds=4.0)
t = np.arange(len(voce))/sr
contaminata = voce + 0.06*np.sin(2*np.pi*2500*t)
sos_notch = al.design_filter("notch", 2500, sr=sr, q=30)
sos_basso = al.design_filter("lowpass", 2000, sr=sr, order=4)
ripulita = al.filter_sos(sos_notch, contaminata)
alternativa = al.filter_sos(sos_basso, contaminata)
versioni = [("Voce di riferimento", voce), ("Con fischio", contaminata),
            ("Notch 2500 Hz", ripulita), ("Passa-basso 2000 Hz", alternativa)]
spettri = [signal.spectrogram(y, fs=sr, window="hann", nperseg=1024,
            noverlap=768, scaling="density") for _, y in versioni]
riferimento = max(float(S.max()) for _, _, S in spettri)
fig = make_subplots(rows=2, cols=2, subplot_titles=[nome for nome, _ in versioni])
for i, (f, tempi, S) in enumerate(spettri):
    fig.add_trace(go.Heatmap(x=tempi, y=f,
        z=10*np.log10(np.maximum(S/riferimento, 1e-8)), coloraxis="coloraxis"),
        row=i//2+1, col=i%2+1)
fig.update_xaxes(title_text="Tempo [s]")
fig.update_yaxes(range=[0, 6000], title_text="Frequenza [Hz]")
fig.update_layout(height=600, coloraxis=dict(colorscale="Magma", cmin=-70, cmax=0,
    colorbar=dict(title="PSD relativa [dB]")), title="Stessa analisi e stessa scala per tutte le versioni")
fig.show()
al.compare_audio(versioni, sr)
